In [ ]:
# [LOCAL-2026] NOT KAGGLE — reload from disk if you still see /kaggle below.
# Local run — الداتا: data/brain_mri (من archive.zip)

import os
import numpy as np
import pandas as pd
from pathlib import Path

def project_root() -> Path:
    """يجد جذر المشروع حتى لو Jupyter شغال من مجلد تاني."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents[:10]]:
        train_g = p / "data" / "brain_mri" / "train" / "glioma"
        if train_g.is_dir():
            return p
    raise FileNotFoundError(
        "لم أجد data/brain_mri/train — فك archive.zip إلى data/brain_mri أو افتح الـnotebook من جذر المشروع."
    )

ROOT = project_root()
DATA_ROOT = ROOT / "data" / "brain_mri"
n_jpg = sum(1 for _ in DATA_ROOT.rglob("*.jpg"))
print("ROOT =", ROOT)
print("Data (jpg count):", n_jpg)


In [ ]:
import os
import shutil
import random
from pathlib import Path
from tqdm import tqdm

random.seed(42)

def project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents[:10]]:
        if (p / "data" / "brain_mri" / "train" / "glioma").is_dir():
            return p
    raise FileNotFoundError("data/brain_mri غير موجودة — راجع الخلية الأولى.")

ROOT = project_root()
DATA_DIR = ROOT / "data" / "brain_mri"
WORK_DIR = ROOT / "data" / "brain_mri_split"

ORIG_TRAIN_DIR = DATA_DIR / "train"
ORIG_TEST_DIR = DATA_DIR / "test"
NEW_TRAIN_DIR = WORK_DIR / "train"
NEW_VAL_DIR = WORK_DIR / "val"
NEW_TEST_DIR = WORK_DIR / "test"

classes = ["glioma", "meningioma", "pituitary", "notumor"]

for target_dir in [NEW_TRAIN_DIR, NEW_VAL_DIR, NEW_TEST_DIR]:
    target_dir.mkdir(parents=True, exist_ok=True)
    for cls in classes:
        (target_dir / cls).mkdir(parents=True, exist_ok=True)

all_class_files = {}
for cls in classes:
    train_imgs = os.listdir(ORIG_TRAIN_DIR / cls)
    test_imgs = os.listdir(ORIG_TEST_DIR / cls)
    all_imgs = [str(ORIG_TRAIN_DIR / cls / f) for f in train_imgs] + [
        str(ORIG_TEST_DIR / cls / f) for f in test_imgs
    ]
    all_class_files[cls] = all_imgs

for cls in tqdm(classes, desc="Processing classes"):
    imgs = all_class_files[cls]
    random.shuffle(imgs)
    n_total = len(imgs)
    n_train = int(n_total * 0.7)
    n_val = int(n_total * 0.15)
    n_test = n_total - n_train - n_val
    train_imgs = imgs[:n_train]
    val_imgs = imgs[n_train : n_train + n_val]
    test_imgs = imgs[n_train + n_val :]
    for f in train_imgs:
        shutil.copy(f, NEW_TRAIN_DIR / cls / os.path.basename(f))
    for f in val_imgs:
        shutil.copy(f, NEW_VAL_DIR / cls / os.path.basename(f))
    for f in test_imgs:
        shutil.copy(f, NEW_TEST_DIR / cls / os.path.basename(f))

print("Done. Splits at:", WORK_DIR)


In [ ]:
import os
from pathlib import Path

def project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents[:10]]:
        if (p / "data" / "brain_mri_split" / "train").is_dir():
            return p
    raise FileNotFoundError("شغّل خلية التقسيم أولاً (data/brain_mri_split).")

ROOT = project_root()
base_dir = str(ROOT / "data" / "brain_mri_split")
splits = ["train", "val", "test"]
classes = ["glioma", "meningioma", "pituitary", "notumor"]

for split in splits:
    print(f"\n{split.upper()} SET:")
    split_total = 0
    for cls in classes:
        folder = os.path.join(base_dir, split, cls)
        if os.path.exists(folder):
            count = len(os.listdir(folder))
            split_total += count
            print(f"  {cls}: {count} images")
        else:
            print(f"  {cls}: Folder not found")
    print(f"  Total {split} images: {split_total}")


In [ ]:
import os
from pathlib import Path

def project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents[:10]]:
        if (p / "data" / "brain_mri_split").is_dir():
            return p
    raise FileNotFoundError("data/brain_mri_split غير موجود.")

ROOT = project_root()
WORK = ROOT / "data" / "brain_mri_split"

for split in ["train", "val", "test"]:
    print(f"\n{split.upper()} directory contents:")
    path = WORK / split
    if path.exists():
        sub = os.listdir(path)
        print(sub[:8], "...")
        first_class = sub[0]
        sample_files = os.listdir(path / first_class)[:5]
        print(f"Sample files in {first_class}: {sample_files}")
    else:
        print("Directory not found.")


In [ ]:
# Ensure TensorFlow is using the GPU optimally
import tensorflow as tf

# Print GPU details
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("GPUs available:", gpus)
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPU found. Training will run on CPU.")


In [ ]:
# Directories for ImageDataGenerator (after split cell)
from pathlib import Path

def project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents[:10]]:
        if (p / "data" / "brain_mri_split" / "train").is_dir():
            return p
    raise FileNotFoundError("شغّل خلية التقسيم أولاً.")

_split = project_root() / "data" / "brain_mri_split"
train_dir = str(_split / "train")
val_dir = str(_split / "val")
test_dir = str(_split / "test")
print(train_dir, val_dir, test_dir, sep="\n")


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SIZE = (224, 224)  # VGG16 input size
BATCH_SIZE = 32

# Training Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation & Test Data: Only rescale
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Flow from directories
train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
)
val_gen = val_test_datagen.flow_from_directory(
    val_dir, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
)
test_gen = val_test_datagen.flow_from_directory(
    test_dir, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)


In [ ]:
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import VGG16

# Load VGG16 base (excluding top output layers), load pretrained weights
base_model = VGG16(weights='imagenet', include_top=False, input_shape=IMAGE_SIZE + (3,))
base_model.trainable = False  # Freeze the base initially

# Add advanced classification layers
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

# Compile with advanced optimizer
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Get class indices and ground-truth labels from generator
labels = train_gen.classes
class_labels = np.unique(labels)
weights = compute_class_weight(class_weight='balanced', classes=class_labels, y=labels)
class_weight_dict = dict(zip(class_labels, weights))

# Pass this dictionary (not a string) to class_weight
history = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=[checkpoint_cb, earlystop_cb, reduce_lr_cb],
    class_weight=class_weight_dict
)


In [ ]:
# Unfreeze last convolutional blocks for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(1e-5),  # Lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Fine-tuning
fine_tune_epochs = 20
history_finetune = model.fit(
    train_gen,
    epochs=fine_tune_epochs,
    validation_data=val_gen,
    callbacks=[checkpoint_cb, earlystop_cb, reduce_lr_cb]
)


In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(test_gen, verbose=2)
print(f'Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}')


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Get true labels and predictions
Y_true = test_gen.classes
Y_pred_proba = model.predict(test_gen)
Y_pred = np.argmax(Y_pred_proba, axis=1)

# Labels from generator
target_names = list(test_gen.class_indices.keys())

print(classification_report(Y_true, Y_pred, target_names=target_names))

# Optional: Confusion Matrix
import seaborn as sns
import matplotlib.pyplot as plt
cm = confusion_matrix(Y_true, Y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Data from your training log
epochs = list(range(1, 21))
training_accuracy = [0.7309, 0.8084, 0.8484, 0.8622, 0.8845, 0.8954, 0.9028, 0.9096, 0.9187, 0.9246, 0.9322, 0.9317, 0.9426, 0.9509, 0.9507, 0.9564, 0.9602, 0.9631, 0.9654, 0.9652]
validation_accuracy = [0.8451, 0.8787, 0.8881, 0.8927, 0.8918, 0.9002, 0.9132, 0.8993, 0.9095, 0.9282, 0.9142, 0.9076, 0.9422, 0.944, 0.9319, 0.9384, 0.9618, 0.9515, 0.958, 0.9571]
training_loss = [0.6818, 0.5115, 0.4092, 0.3747, 0.3506, 0.2876, 0.2685, 0.2496, 0.2307, 0.2129, 0.1997, 0.1805, 0.1563, 0.139, 0.1475, 0.1185, 0.1089, 0.114, 0.0937, 0.0914]
validation_loss = [0.4425, 0.3801, 0.3595, 0.356, 0.3119, 0.2729, 0.2403, 0.2844, 0.2619, 0.1854, 0.2703, 0.2505, 0.1522, 0.1434, 0.1762, 0.1629, 0.1116, 0.1547, 0.1232, 0.1156]

plt.figure(figsize=(14, 6))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(epochs, training_accuracy, marker='o', label='Training Accuracy')
plt.plot(epochs, validation_accuracy, marker='o', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(epochs, training_loss, marker='o', label='Training Loss')
plt.plot(epochs, validation_loss, marker='o', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# Data from your results
epochs = list(range(1, 21))
training_accuracy = [0.7309, 0.8084, 0.8484, 0.8622, 0.8845, 0.8954, 0.9028, 0.9096, 0.9187, 0.9246, 0.9322, 0.9317, 0.9426, 0.9509, 0.9507, 0.9564, 0.9602, 0.9631, 0.9654, 0.9652]
validation_accuracy = [0.8451, 0.8787, 0.8881, 0.8927, 0.8918, 0.9002, 0.9132, 0.8993, 0.9095, 0.9282, 0.9142, 0.9076, 0.9422, 0.944, 0.9319, 0.9384, 0.9618, 0.9515, 0.958, 0.9571]

# Create DataFrame
df = pd.DataFrame({
    'Epoch': epochs,
    'Training Accuracy (%)': [f"{a*100:.2f}" for a in training_accuracy],
    'Validation Accuracy (%)': [f"{a*100:.2f}" for a in validation_accuracy],
})

# Jupyter/Kaggle auto-renders as a table
df
